In [1]:
!python -m pip install --upgrade pip

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.1
    Uninstalling pip-25.1:
      Successfully uninstalled pip-25.1


In [2]:
pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install --upgrade numexpr

  Attempting uninstall: numexpr
    Found existing installation: numexpr 2.10.1
    Uninstalling numexpr-2.10.1:
      Successfully uninstalled numexpr-2.10.1
Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
BASE_URL = "https://books.toscrape.com/catalogue/"
books_data = []

In [5]:
def get_rating(word):
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(word, 0)

In [6]:
def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        rating_word = book.find("p", class_="star-rating")["class"][1]
        rating = get_rating(rating_word)

        books_data.append({
            "Title": title,
            "Price (£)": price,
            "Rating": rating
        })

    # Follow "next" page
    next_btn = soup.find("li", class_="next")
    if next_btn:
        next_page = BASE_URL + next_btn.a["href"]
        scrape_page(next_page)

In [7]:
# Start scraping
scrape_page("https://books.toscrape.com/catalogue/page-1.html")

In [8]:
# Save to CSV
df = pd.DataFrame(books_data)
df.to_csv("books.csv", index=False)
print(f"Done! Scraped {len(df)} books.")
print(df.head())

Done! Scraped 1000 books.
                                   Title Price (£)  Rating
0                   A Light in the Attic   Â£51.77       3
1                     Tipping the Velvet   Â£53.74       1
2                             Soumission   Â£50.10       1
3                          Sharp Objects   Â£47.82       4
4  Sapiens: A Brief History of Humankind   Â£54.23       5


In [9]:
import pandas as pd
df = pd.read_csv("books.csv")
print(df.shape)        # (1000, 3)
print(df.describe())   # Stats on price/rating

(1000, 3)
            Rating
count  1000.000000
mean      2.923000
std       1.434967
min       1.000000
25%       2.000000
50%       3.000000
75%       4.000000
max       5.000000
